# 02 — Exploratory Data Analysis
Label distribution, signal distributions, missing value and duplicate checks.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from signalrag.config import TRAIN_CSV, TEST_CSV, FEATURE_COLS, LABEL_COL


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")


## Null / duplicate check

In [ ]:
print("Nulls (train):", train_df.isnull().sum().sum())
print("Duplicates (train):", train_df.duplicated().sum())
print(train_df.dtypes)


## Label distribution

In [ ]:
label_counts = train_df[LABEL_COL].value_counts()
fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(label_counts.values, labels=["Hallucinated", "Faithful"],
       colors=["#eb6834", "#2a78d6"], autopct="%1.1f%%", startangle=90)
ax.set_title("Label Distribution — Train Set", fontsize=12)
plt.tight_layout()
plt.savefig("eda_label_distribution.png", dpi=300, bbox_inches="tight")
plt.show()


## Signal distributions (test set)

In [ ]:
faithful     = test_df[test_df[LABEL_COL] == 1]
hallucinated = test_df[test_df[LABEL_COL] == 0]
rows = []
for sig in FEATURE_COLS:
    if sig not in test_df.columns:
        continue
    f, h = faithful[sig].dropna(), hallucinated[sig].dropna()
    d = abs(f.mean() - h.mean())
    rows.append({"Signal": sig, "Faithful": f.mean(), "Hallucinated": h.mean(),
                 "Delta": d, "Discriminative": "Yes" if d > 0.05 else "Weak"})
dist_df = pd.DataFrame(rows).sort_values("Delta", ascending=False)
print(dist_df.to_string(index=False))
